In [1]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import os

def generate_refined_keyword_wordcloud(file1_path, file2_path, file3_path, save_path):
    """
    合并三个txt文件的关键词，精细化分类后生成词云图
    :param file1_path: 第一个关键词txt文件路径
    :param file2_path: 第二个关键词txt文件路径
    :param file3_path: 第三个关键词txt文件路径
    :param save_path: 词云图保存路径（含文件名，如"refined_wordcloud.png"）
    :return: 去重关键词列表、精细化分类字典
    """
    # 1. 读取单个文件关键词的工具函数
    def read_keywords(file_path):
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"文件不存在：{file_path}")
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            # 兼容换行/空格分割，过滤空字符串
            keywords = [kw.strip() for kw in content.replace('\n', ' ').split() if kw.strip()]
        return keywords
    
    # 读取三个文件
    kw1 = read_keywords(file1_path)
    kw2 = read_keywords(file2_path)
    kw3 = read_keywords(file3_path)
    
    # 2. 合并去重
    all_keywords = list(set(kw1 + kw2 + kw3))
    if not all_keywords:
        raise ValueError("三个文件中未提取到有效关键词")
    
    # 3. 精细化分类（15大类）
    refined_category = {
        # 人物相关（3子类）
        "人物-亲属关系": ["丈夫", "妻子", "妈妈", "奶奶", "爷爷", "爸爸", "儿子", "女儿", "双胞胎", "姐弟恋", "父母", "爸妈", "家属"],
        "人物-社会身份": ["学生", "老师", "医生", "演员", "博主", "网红", "爱豆", "骑手", "导购", "客服", "女房东", "投资者", "高官", "法官", "主刀", "教授", "导师", "小学老师"],
        "人物-群体标签": ["儿童", "婴儿", "孕妇", "女生", "年轻人", "留子", "陌生人", "民众", "活人", "继承人", "娃妈", "女巫", "女性", "女方", "女演员", "男方", "男演员"],
        
        # 生活场景（3子类）
        "生活-日常活动": ["上班", "下班", "上学", "回家", "回村", "逛街", "睡觉", "补觉", "吃饭", "做饭", "洗碗", "洗澡", "洗衣", "通勤", "早市", "早餐", "散步", "走路", "走过"],
        "生活-消费购物": ["卖货", "回购", "购物", "开箱", "外卖", "快递", "小卖部", "商店", "商家", "超市", "同款", "吊牌", "衣服", "围巾", "帽子", "包包", "化妆品", "护肤品", "零食", "块钱"],
        "生活-休闲健康": ["健身", "锻炼", "偶遇", "旅游", "庆生", "奔现", "约会", "婚礼", "婚房", "养猫", "养狗", "宠物", "寄养", "小猫", "狗狗", "浣熊", "小猪", "爬行动物"],
        
        # 社会事件（3子类）
        "社会-安全事故": ["事故", "坠亡", "火灾", "倒塌", "触电", "车祸", "溺水", "窒息", "死亡", "身亡", "脑死亡", "坠楼", "爆炸", "踩踏"],
        "社会-情感纠纷": ["离婚", "结婚", "订婚", "分手", "失恋", "求婚", "出轨", "捉奸", "争吵", "暧昧", "冷战", "恋情", "情侣", "夫妻", "恋爱", "恋童", "情房", "情趣"],
        "社会-职场学业": ["失业", "离职", "辞职", "考编", "保研", "留学", "考试", "重考", "招聘", "兼职", "实习", "作业", "毕业", "论文", "读书", "考研", "读研", "研究生"],
        
        # 娱乐影视（2子类）
        "娱乐-明星艺人": ["热巴", "王源", "陈松伶", "黄景", "弦子", "爱豆", "粉丝", "站姐", "私联", "经纪", "解约", "签约", "演员", "演唱会", "红毯", "造型", "妆造"],
        "娱乐-影视综艺": ["剧组", "综艺", "短剧", "电影", "电视剧", "首映", "上映", "打戏", "生图", "杀青", "大结局", "娱乐圈", "开播", "停播", "播出", "剧情", "角色", "配角"],
        
        # 产品物品（2子类）
        "物品-家电数码": ["电动车", "电饭煲", "电热毯", "手机", "电脑", "相机", "手表", "洗衣机", "冰箱", "空调", "音响", "机器人", "电动", "手机", "电视", "电梯", "电梯门"],
        "物品-食品饮品": ["牛奶", "啤酒", "劲酒", "醪糟", "米饭", "五花肉", "大闸蟹", "鸡蛋", "水果", "蔬菜", "零食", "泡面", "奶皮子", "奶酒", "咖啡", "烈酒", "椰汁", "奶茶"],
        
        # 健康医疗（2子类）
        "健康-疾病症状": ["咳嗽", "炎症", "痔疮", "流感", "甲流", "发烧", "感冒", "腹泻", "胃痛", "头痛", "牙痛", "肺癌", "脑死亡", "月经", "体寒", "体热", "脱发", "失眠", "过敏", "细菌"],
        "健康-医疗行为": ["手术", "打针", "吃药", "体检", "堕胎", "产检", "陪产", "疗伤", "住院", "出院", "化疗", "放疗", "针灸", "推拿", "按摩", "急救", "急诊", "尸检", "检查", "查出"],
        
        # 其他
        "其他-特殊概念": ["A股", "国标", "地铁", "飞机", "全球", "宇宙", "时代", "法律", "政策", "规定", "禁令", "公示", "公开", "官方", "公司", "集团", "国家", "地球", "广场", "公园"]
    }
    
    # 分配未归类关键词到"其他-特殊概念"
    categorized_kws = [kw for cat in refined_category.values() for kw in cat]
    for kw in all_keywords:
        if kw not in categorized_kws:
            refined_category["其他-特殊概念"].append(kw)
    
    # 4. 生成词云（优化配色与字体）
    keyword_text = ' '.join(all_keywords)
    # 适配Windows/Mac/Linux中文字体
    font_candidates = [
        'C:/Windows/Fonts/simhei.ttf',  # Windows 黑体
        '/System/Library/Fonts/PingFang.ttc',  # Mac 苹方
        '/usr/share/fonts/wenquanyi/wqy-zenhei/wqy-zenhei.ttc'  # Linux 文泉驿
    ]
    font_path = None
    for path in font_candidates:
        if os.path.exists(path):
            font_path = path
            break
    if not font_path:
        raise FileNotFoundError("未找到系统中文字体，请手动指定font_path参数")
    
    # 词云样式配置（暖色调主题，适配微博热搜氛围）
    wordcloud = WordCloud(
        width=1600,
        height=1000,
        background_color='#f5f5f5',
        font_path=font_path,
        max_words=300,
        font_step=2,
        random_state=42,
        collocations=False,
        prefer_horizontal=0.8,
        color_func=lambda *args, **kwargs: np.random.choice(['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6'])
    ).generate(keyword_text)
    
    # 5. 保存词云图（高分辨率）
    plt.figure(figsize=(20, 12), dpi=300)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('微博热搜关键词词云（精细化分类）', fontsize=24, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=300, facecolor='#f5f5f5')
    plt.close()
    
    # 输出分类统计信息
    print("="*50)
    print("精细化分类统计结果：")
    print("="*50)
    total_categorized = 0
    for cat, kws in refined_category.items():
        print(f"{cat}：{len(kws)}个关键词")
        total_categorized += len(kws)
    print(f"="*50)
    print(f"去重后总关键词数：{len(all_keywords)}")
    print(f"分类覆盖关键词数：{total_categorized}")
    print(f"词云图已保存至：{save_path}")
    print("="*50)
    
    return all_keywords, refined_category

# 示例调用（替换为实际文件路径）
if __name__ == "__main__":
    _, categories = generate_refined_keyword_wordcloud(
        file1_path="D:/Jupyter Code/数据采集期末论文/2025.12.04.1030微博数据/可视化数据test/accurate_keywords.txt",  # 第一个关键词文件
        file2_path="D:/Jupyter Code/数据采集期末论文/2025.12.03.2118微博数据/可视化数据/accurate_keywords.txt",  # 第二个关键词文件
        file3_path="D:/Jupyter Code/数据采集期末论文/2025.12.03.0942微博数据/可视化数据/accurate_keywords.txt",  # 第三个关键词文件
        save_path="D:/Jupyter Code/数据采集期末论文/refined_weibo_wordcloud.png"  
    )

C:\Users\fybollOvO\AppData\Local\Temp\ipykernel_73548\1188443848.py:110: UserWarning: Glyph 24494 (\N{CJK UNIFIED IDEOGRAPH-5FAE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\fybollOvO\AppData\Local\Temp\ipykernel_73548\1188443848.py:110: UserWarning: Glyph 21338 (\N{CJK UNIFIED IDEOGRAPH-535A}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\fybollOvO\AppData\Local\Temp\ipykernel_73548\1188443848.py:110: UserWarning: Glyph 28909 (\N{CJK UNIFIED IDEOGRAPH-70ED}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\fybollOvO\AppData\Local\Temp\ipykernel_73548\1188443848.py:110: UserWarning: Glyph 25628 (\N{CJK UNIFIED IDEOGRAPH-641C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\fybollOvO\AppData\Local\Temp\ipykernel_73548\1188443848.py:110: UserWarning: Glyph 20851 (\N{CJK UNIFIED IDEOGRAPH-5173}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\fybollOvO\AppData\Local\Temp\ipykernel_73548\1188443848.py:11

精细化分类统计结果：
人物-亲属关系：13个关键词
人物-社会身份：18个关键词
人物-群体标签：17个关键词
生活-日常活动：19个关键词
生活-消费购物：20个关键词
生活-休闲健康：18个关键词
社会-安全事故：14个关键词
社会-情感纠纷：18个关键词
社会-职场学业：18个关键词
娱乐-明星艺人：17个关键词
娱乐-影视综艺：18个关键词
物品-家电数码：17个关键词
物品-食品饮品：18个关键词
健康-疾病症状：20个关键词
健康-医疗行为：20个关键词
其他-特殊概念：621个关键词
去重后总关键词数：807
分类覆盖关键词数：886
词云图已保存至：D:/Jupyter Code/数据采集期末论文/refined_weibo_wordcloud.png
